# Fase 3: Modelo Avanzado con XGBoost e Interpretabilidad SHAP
## Diagnóstico Explicable y Detección de Desgaste en Máquina CNC

**Autor:** Íñigo Guzmán  

### Objetivo
Entrenar un modelo basado en Gradient Boosting (**XGBoost**) regularizado para predecir el desgaste de la fresa a partir de las señales dinámicas de corte, y utilizar **SHAP (SHapley Additive exPlanations)** para transformar el modelo en una caja blanca auditable en entornos de mantenimiento predictivo industrial.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import json

import pandas as pd
from IPython.display import Image

reports_dir = Path("../reports")
figures_dir = reports_dir / "figures"
models_dir = Path("../models")

with open(reports_dir / "xgb_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

pd.DataFrame([metrics], index=["XGBoost (GroupKFold)"])

### 1. Importancia Global de Variables (SHAP Beeswarm Plot)
El gráfico *beeswarm* muestra cómo cada variable empuja la predicción hacia herramienta desgastada (*worn*, valores SHAP positivos) o nueva (*unworn*, valores SHAP negativos):
- **Corriente del husillo (`S1_OutputCurrent_mean`, `S1_CurrentFeedback_rms`):** Valores altos de consumo eléctrico en el cabezal de giro correlacionan fuertemente con desgaste debido a la fricción de los filos romos.
- **Fuerzas de empuje en Z (`Z1_ActualAcceleration`, `Z1_CurrentFeedback`):** El incremento en la fuerza vertical refleja la pérdida de capacidad de penetración de la herramienta en el material.

In [ ]:
Image(filename=str(figures_dir / "shap_summary_beeswarm.png"))

### 2. Ranking de Importancia Absoluta (Top 10)
El siguiente gráfico clasifica los sensores por su impacto medio absoluto en las decisiones del modelo:

In [ ]:
Image(filename=str(figures_dir / "shap_bar_importance.png"))

### 3. Conclusiones y Preparación para Despliegue (Fase 4)
- El modelo entrenado y sus metadatos de entrada han sido serializados en `models/cnc_tool_wear_xgb.pkl` y `models/feature_columns.json`.
- En la Fase 4, desplegaremos este modelo en una interfaz interactiva de **Streamlit** para permitir a los operarios cargar archivos de telemetría de mecanizado y visualizar en tiempo real el diagnóstico y las explicaciones de SHAP.